In [ ]:
# Install
!pip install torch -q


In [1]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import requests

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# Load dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


In [4]:
# Batch loader
block_size = 64
batch_size = 32

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [5]:
# Transformer model
class TransformerModel(nn.Module):
    def __init__(self, d_model, n_layers):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, block_size, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=4*d_model,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        x = self.token_emb(x) + self.pos_emb[:, :T, :]
        x = self.transformer(x)
        x = self.ln(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))

        return logits, loss

In [10]:
# Train function
def train_model(model, steps=500):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    losses = []
    start = time.time()

    for step in range(steps):
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if step % 50 == 0:
            print(f"Step {step} | Loss {loss.item():.4f}")

    return losses, time.time() - start


In [11]:
# Create models
model_wide = TransformerModel(d_model=512, n_layers=2)   # Wide
model_deep = TransformerModel(d_model=256, n_layers=8)   # Deep

In [12]:
# Train both
print("Training Wide Model (2 layers, d=512)")
loss_wide, time_wide = train_model(model_wide)

print("\nTraining Deep Model (8 layers, d=256)")
loss_deep, time_deep = train_model(model_deep)

Training Wide Model (2 layers, d=512)
Step 0 | Loss 4.3503
Step 50 | Loss 2.4988
Step 100 | Loss 2.3518
Step 150 | Loss 2.3172
Step 200 | Loss 2.2986
Step 250 | Loss 2.2442
Step 300 | Loss 2.2000
Step 350 | Loss 2.1784
Step 400 | Loss 2.1564
Step 450 | Loss 2.2079

Training Deep Model (8 layers, d=256)
Step 0 | Loss 4.2193
Step 50 | Loss 2.5158
Step 100 | Loss 2.3518
Step 150 | Loss 2.2913
Step 200 | Loss 2.2605
Step 250 | Loss 2.2316
Step 300 | Loss 2.2272
Step 350 | Loss 2.1656
Step 400 | Loss 2.1933
Step 450 | Loss 2.1655


In [13]:
# Results
print("\nFINAL RESULTS: ")
print(f"Wide Model Loss: {loss_wide[-1]:.4f}")
print(f"Deep Model Loss: {loss_deep[-1]:.4f}")
print(f"Wide Time: {time_wide:.2f}s")
print(f"Deep Time: {time_deep:.2f}s")


FINAL RESULTS: 
Wide Model Loss: 2.1779
Deep Model Loss: 2.1385
Wide Time: 1032.73s
Deep Time: 1277.13s
